# 03 — M2: VGG-like sâu + BN + residual + SpatialDropout + GAP

**Chủ: Hoàng** · lý thuyết: `docs/LY_THUYET.md` 2.2–2.6 · code: `src/gtsrb/models/m2_vggres.py`

Notebook này chứng minh **bằng số** rằng từng thành phần kiến trúc đáng giá bao nhiêu —
không nói theo bài báo, mà đo trên chính dữ liệu của nhóm.

Train trước: `make train-m2`

In [ ]:
# Cho notebook thấy src/.  Chạy ô này đầu tiên.
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import os; os.chdir(ROOT)          # mọi đường dẫn trong repo là tương đối từ gốc

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from gtsrb import CLASS_NAMES, NUM_CLASSES
print("torch", torch.__version__, "| MPS", torch.backends.mps.is_available(),
      "| CUDA", torch.cuda.is_available())

In [ ]:
# Tiện ích dùng chung: tìm và nạp lại một run đã train
import glob, json
sys.path.insert(0, str(ROOT / "scripts"))
from evaluate import load_run, test_loader_for
from gtsrb.utils.seed import pick_device

device = pick_device("auto")

def find_runs(prefix=""):
    """Danh sách thư mục run có checkpoint, lọc theo tiền tố tên model."""
    dirs = [pathlib.Path(p) for p in sorted(glob.glob("artifacts/runs/*"))]
    return [d for d in dirs if (d / "best.pt").exists() and d.name.startswith(prefix)]

def latest(prefix):
    runs = find_runs(prefix)
    if not runs:
        raise FileNotFoundError(
            f"Chưa có run nào tên bắt đầu bằng {prefix!r}. Chạy: make train-{prefix[:2]}")
    return runs[-1]

print("Các run đã có:")
for d in find_runs(): print("  ", d.name)

## 1. Kiến trúc

```
Stem      Conv(3→32,3×3)+BN+ReLU             (B,  32, 48, 48)
Stage 1   ResBlock(32→32)   → Pool → SD(0,1) (B,  32, 24, 24)
Stage 2   ResBlock(32→64)   → Pool → SD(0,2) (B,  64, 12, 12)
Stage 3   ResBlock(64→128)  → Pool → SD(0,3) (B, 128,  6,  6)
Stage 4   ResBlock(128→256) → Pool → SD(0,3) (B, 256,  3,  3)
Head      GAP → Dropout(0,5) → FC(256→43)    (B,  43)
```
`SD` = `nn.Dropout2d` (spatial dropout — bỏ **cả kênh**, không phải từng pixel).

In [ ]:
from gtsrb.models.registry import build_model, count_parameters, parameter_table

m1 = build_model("m1_lenet",  img_size=48)
m2 = build_model("m2_vggres", img_size=48)

n_conv = lambda m: sum(1 for x in m.modules() if isinstance(x, torch.nn.Conv2d))
compare = pd.DataFrame({
    "model": ["M1 LeNet (Flatten+FC)", "M2 VGG-res (GAP)"],
    "số lớp conv": [n_conv(m1), n_conv(m2)],
    "tổng tham số": [count_parameters(m1), count_parameters(m2)],
})
compare["so với M1"] = (compare["tổng tham số"] / count_parameters(m1) * 100).round(1).astype(str) + "%"
print(compare.to_string(index=False))
print()
print("M2 sâu hơn nhưng ÍT tham số hơn. Lớp đắt nhất của mỗi model:")
for name, m in [("M1", m1), ("M2", m2)]:
    top = parameter_table(m).iloc[0]
    print(f"  {name}: {top['layer']:24s} {top['type']:8s} {top['params']:>9,}  ({top['percent']:.1f}%)")

**Đây là một trong những con số thuyết phục nhất của cả dự án.**
M2 có 9 lớp conv so với 2 của M1, mà tổng tham số chỉ bằng ~51%.

Hai lý do:
1. Conv dùng **tham số chia sẻ** — một kernel 3×3 trượt khắp ảnh, nên mỗi lớp conv rẻ.
2. M1 dồn 97,3% tham số vào một lớp `Flatten(9216)→FC(256)`; M2 thay chỗ đó bằng
   `GlobalAvgPool→FC(256→43)` chỉ 11.051 tham số.

**Bài học: số lớp KHÔNG tỉ lệ với số tham số. Chỗ đắt là lớp fully-connected.**

## 2. Residual block — nhìn vào code

```python
def forward(self, x):
    identity = self.skip(x)                       # Identity, hoặc Conv1x1+BN khi đổi kênh
    out = self.relu(self.bn1(self.conv1(x)))
    out = self.bn2(self.conv2(out))
    return self.relu(out + identity)              # ← cộng TRƯỚC, ReLU SAU
```

Thứ tự `cộng rồi ReLU` là đúng theo bài ResNet gốc. Làm ngược lại (`ReLU rồi cộng`)
thì nhánh skip bị chặn ở 0 và mất tác dụng "đường cao tốc gradient".

Vì sao `bias=False` ở conv: ngay sau là BatchNorm, mà BN có `beta` làm đúng việc của
bias — thêm bias là thừa tham số và không tác dụng gì.

In [ ]:
from gtsrb.models.m2_vggres import ResidualBlock

block = ResidualBlock(32, 64)
print(block)
x = torch.randn(2, 32, 24, 24)
print(f"\nvào  {tuple(x.shape)}  ->  ra {tuple(block(x).shape)}")
print("skip dùng:", type(block.skip).__name__,
      "(Conv1x1+BN vì số kênh đổi 32->64)")

same = ResidualBlock(64, 64)
print("khi KHÔNG đổi kênh, skip dùng:", type(same.skip).__name__, "(không tốn tham số)")

## 3. ★ Ablation nội bộ: từng thành phần đáng giá bao nhiêu?

Ba cờ `use_bn`, `use_residual`, `use_spatial_dropout` cho phép bỏ từng thành phần
mà **giữ nguyên mọi thứ khác** — đúng nguyên tắc một-biến-một-lần.

Chạy thật: `python scripts/run_ablation.py --axes components`

In [ ]:
variants = {
    "đầy đủ":                 dict(),
    "bỏ BatchNorm":           dict(use_bn=False),
    "bỏ residual":            dict(use_residual=False),
    "bỏ spatial dropout":     dict(use_spatial_dropout=False),
}
rows = []
for label, kwargs in variants.items():
    m = build_model("m2_vggres", img_size=48, **kwargs)
    rows.append({"biến thể": label, "tham số": count_parameters(m),
                 "số lớp conv": n_conv(m)})
print(pd.DataFrame(rows).to_string(index=False))
print()
print("Lưu ý: bỏ BN làm TĂNG nhẹ tham số (conv phải thêm bias trở lại).")
print("Bỏ residual cũng đổi số tham số (không còn conv 1x1 projection).")
print("Bỏ spatial dropout KHÔNG đổi tham số (dropout không có tham số học được)")
print("-> nên tác dụng của nó chỉ thấy được qua accuracy, không qua số tham số.")

In [ ]:
# Nạp bảng ablation nếu đã chạy
abl_path = pathlib.Path("reports/tables/ablation.csv")
if abl_path.exists():
    abl = pd.read_csv(abl_path)
    comp = abl[abl["tag"].astype(str).str.startswith("components")]
    if len(comp):
        print(comp[["tag", "val_macro_f1", "test_macro_f1", "test_top1"]].to_string(index=False))
    else:
        print("Chưa có run nào thuộc trục 'components'.")
        print("Chạy: python scripts/run_ablation.py --axes components")
else:
    print("Chưa có reports/tables/ablation.csv")
    print("Chạy: python scripts/run_ablation.py --axes components")
    print("      python scripts/evaluate.py --runs \"artifacts/runs/*\"")
    print("      python scripts/run_ablation.py --collect-only")

## 4. Model scaling: width và depth

In [ ]:
rows = []
for w in (0.5, 1.0, 2.0):
    m = build_model("m2_vggres", img_size=48, width_mult=w)
    rows.append({"trục": "width", "giá trị": w, "tham số": count_parameters(m)})
for d in (3, 4, 5):
    m = build_model("m2_vggres", img_size=48, n_stages=d)
    rows.append({"trục": "depth (số stage)", "giá trị": d, "tham số": count_parameters(m)})
scaling = pd.DataFrame(rows)
print(scaling.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, axis_name in zip(axes, ["width", "depth (số stage)"]):
    part = scaling[scaling["trục"] == axis_name]
    ax.plot(part["giá trị"], part["tham số"] / 1e6, marker="o")
    ax.set_xlabel(axis_name); ax.set_ylabel("triệu tham số")
    ax.set_title(f"Scaling theo {axis_name}"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()
print("\nWidth tăng x2 -> tham số tăng ~x4 (vì conv có C_in * C_out).")
print("Depth +1 stage  -> tham số tăng nhiều hơn tuyến tính (stage sâu nhiều kênh hơn).")

## 5. Đường cong huấn luyện và so sánh với M1

In [ ]:
runs_to_plot = {}
for prefix in ("m1_lenet", "m2_vggres"):
    try: runs_to_plot[prefix] = latest(prefix)
    except FileNotFoundError: print(f"chưa có run {prefix}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, run in runs_to_plot.items():
    h = pd.read_csv(run / "train_log.csv")
    axes[0].plot(h.epoch, h.val_top1, label=name)
    axes[1].plot(h.epoch, h.val_macro_f1, label=name)
axes[0].set_title("val top-1"); axes[1].set_title("val macro-F1")
for ax in axes:
    ax.set_xlabel("epoch"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

for name, run in runs_to_plot.items():
    h = pd.read_csv(run / "train_log.csv")
    print(f"{name:12s} best val macro-F1 = {h.val_macro_f1.max():.4f} "
          f"(epoch {int(h.loc[h.val_macro_f1.idxmax(), 'epoch'])}, "
          f"{h.seconds.sum()/60:.1f} phút)")

**Chú ý khi so sánh.** Chênh lệch nhìn thấy trên hình **chưa chắc có ý nghĩa thống kê**.
Trên 12.630 ảnh test, 0,2 điểm top-1 chỉ là ~25 ảnh. Phải dùng **McNemar**
(`notebooks/05`, hoặc `make eval`) để biết chênh lệch là thật hay chỉ là nhiễu.